# French Oral Narrative: Token-Level Coherence Decay

Replicates the RAID v7 / Buckeye fine-grained token-reveal analysis on **spoken French oral narrative** from the French Oral Narrative Corpus (87 stories, 17 storytellers).

**Cross-linguistic test**: Does the ~0.75 power law exponent hold for French?
- English written (RAID): alpha = -0.75
- English spoken (Buckeye): alpha = -0.73
- French spoken: alpha = ???

**Method**: Identical to Buckeye — fixed token-offset target selection, no sentence parsing, fully language-agnostic. Mistral-7B was trained on substantial French text, making it a valid probe for French coherence.

**Two analysis modes**:
1. Per-story (87 documents, 264-6949 words each)
2. Per-storyteller concatenated (17 documents, up to 19K words each)

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import uniform_filter1d
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('Imports OK')

In [ ]:
# === Configuration ===
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/french_oral_processed")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/French_oral_finegrain")
    DRIVE_RAID_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_finegrain")
    DRIVE_BUCKEYE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/Buckeye_finegrain")
    if (DRIVE_DATA / "per_story.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/french_oral_processed")
        if not (LOCAL_DATA / "per_story.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            print("Upload per_story.jsonl:")
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/french_oral_finegrain")
    DATA_DIR = Path("../data/french_oral_processed")
    DRIVE_RAID_RESULTS = Path("../results/raid_finegrain")
    DRIVE_BUCKEYE_RESULTS = Path("../results/Buckeye_finegrain")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True
MAX_CONTEXT = 100       # tokens of context to reveal
TARGET_LEN = 30         # tokens in target region
TARGET_FRACTIONS = [0.25, 0.50, 0.75]
MIN_CONTEXT_BEFORE_TARGET = MAX_CONTEXT + 10
RANDOM_SEED = 42

print(f"Max context: {MAX_CONTEXT} tokens")
print(f"Target length: {TARGET_LEN} tokens")
print(f"Target positions: {TARGET_FRACTIONS}")

In [ ]:
# === Load French oral narrative data ===
# Use per-story documents (87 stories, each a monologue)
corpus = []
with open(DATA_DIR / "per_story.jsonl") as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"Loaded {len(corpus)} story documents")
word_counts = [len(d['text'].split()) for d in corpus]
print(f"Word counts: min={min(word_counts)}, max={max(word_counts)}, mean={np.mean(word_counts):.0f}")
print(f"Total words: {sum(word_counts):,}")

# How many stories have enough context for all 3 target positions?
from transformers import AutoTokenizer as AT
_tok = AT.from_pretrained(MODEL_NAME)
eligible = 0
for doc in corpus:
    n_tok = len(_tok.encode(doc['text'], add_special_tokens=False))
    if int(n_tok * 0.25) >= MIN_CONTEXT_BEFORE_TARGET:
        eligible += 1
print(f"\nStories with enough tokens for 25% target: {eligible}/{len(corpus)}")
del _tok

In [ ]:
# === Load model ===
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )
model.eval()
print("Model loaded")

In [ ]:
# === Core functions ===

@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    """Compute perplexity over target region given preceding context."""
    if target_start >= target_end - 1:
        return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def compute_token_reveal_curve(full_ids, target_start, target_end,
                                shuffled=False, rng_shuf=None):
    """Token-by-token context reveal for a single target region.
    
    No sentence boundary detection needed — context expands backward
    from target_start, one token at a time.
    
    Args:
        full_ids: complete token sequence for the document
        target_start: index of first target token
        target_end: index past last target token
        shuffled: if True, shuffle context tokens (control)
        rng_shuf: random state for shuffling
    """
    target_ids = full_ids[target_start:target_end]
    
    # Context pool = all tokens before the target
    context_pool = list(full_ids[:target_start])
    
    if shuffled and rng_shuf is not None:
        context_pool = list(context_pool)
        rng_shuf.shuffle(context_pool)
    
    max_ctx = min(MAX_CONTEXT, len(context_pool))
    if max_ctx < 10:
        return None
    
    ppls = []
    ctx_lengths = []
    
    for ctx_len in range(1, max_ctx + 1):
        # Take the last ctx_len tokens before target (expanding backward)
        ctx_tokens = context_pool[-ctx_len:]
        
        chunk = ctx_tokens + target_ids
        ppl = compute_ppl(chunk, len(ctx_tokens), len(chunk))
        if not math.isinf(ppl):
            ppls.append(ppl)
            ctx_lengths.append(ctx_len)
    
    if len(ppls) < 10:
        return None
    
    return {
        'ctx_lengths': ctx_lengths,
        'ppls': ppls,
    }


def process_document(doc, rng_shuf):
    """Process one document: sample target positions, compute intact + shuffled curves."""
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    
    intact_curves = []
    shuffled_curves = []
    
    for frac in TARGET_FRACTIONS:
        target_start = int(n * frac)
        target_end = min(target_start + TARGET_LEN, n)
        
        # Need enough context before target
        if target_start < MIN_CONTEXT_BEFORE_TARGET:
            continue
        if target_end - target_start < 5:
            continue
        
        # Intact
        result = compute_token_reveal_curve(full_ids, target_start, target_end,
                                            shuffled=False)
        if result is not None:
            result['doc_id'] = doc['doc_id']
            result['target_frac'] = frac
            intact_curves.append(result)
        
        # Shuffled control
        result_s = compute_token_reveal_curve(full_ids, target_start, target_end,
                                              shuffled=True, rng_shuf=rng_shuf)
        if result_s is not None:
            result_s['doc_id'] = doc['doc_id']
            result_s['target_frac'] = frac
            shuffled_curves.append(result_s)
    
    return intact_curves, shuffled_curves

print("Functions defined")

In [ ]:
# === Run computation (or load cached results) ===
results_path = BASE_DIR / "french_oral_intact_v1.json"
shuffled_path = BASE_DIR / "french_oral_shuffled_v1.json"

if results_path.exists() and shuffled_path.exists():
    with open(results_path) as f:
        all_intact = json.load(f)
    with open(shuffled_path) as f:
        all_shuffled = json.load(f)
    print(f"Loaded {len(all_intact)} intact + {len(all_shuffled)} shuffled curves from cache")
else:
    all_intact = []
    all_shuffled = []
    rng_shuf = np.random.RandomState(RANDOM_SEED + 99)
    
    for doc in tqdm(corpus, desc="Processing stories"):
        intact, shuffled = process_document(doc, rng_shuf)
        all_intact.extend(intact)
        all_shuffled.extend(shuffled)
    
    # Save results
    with open(results_path, 'w') as f:
        json.dump(all_intact, f)
    with open(shuffled_path, 'w') as f:
        json.dump(all_shuffled, f)
    print(f"Computed {len(all_intact)} intact + {len(all_shuffled)} shuffled curves")

# Summary
unique_docs = set(c['doc_id'] for c in all_intact)
print(f"\nUnique stories with curves: {len(unique_docs)}")
print(f"Curves per target position:")
for frac in TARGET_FRACTIONS:
    n = sum(1 for c in all_intact if c['target_frac'] == frac)
    print(f"  {frac:.0%}: {n} curves")

## Analysis: Corrected Power Law Fit

Same pipeline as RAID v7:
- Compute raw perplexity curves (intact and shuffled)
- Take marginals (per-token benefit of additional context)
- Subtract shuffled from intact to get **corrected marginals** (pure coherence signal)
- Fit power law on binned corrected marginals

In [ ]:
# === Analysis functions (identical to RAID v7 / Buckeye) ===
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

def compute_raw_ppl_curve(curves):
    """Mean raw perplexity at each context length (not normalized)."""
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    all_ppl = np.array(all_ppl)
    return np.nanmean(all_ppl, axis=0)

def compute_mean_curve(curves):
    """Normalized mean curve (0-1 scale)."""
    all_norm = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        if ppl[0] - ppl[-1] > 0:
            norm = (ppl[0] - ppl) / (ppl[0] - ppl[-1])
            interp = np.interp(common_x, ctx, norm, left=np.nan, right=np.nan)
            all_norm.append(interp)
    all_norm = np.array(all_norm)
    mean = np.nanmean(all_norm, axis=0)
    sem = np.nanstd(all_norm, axis=0) / np.sqrt(np.sum(~np.isnan(all_norm), axis=0))
    return mean, sem

def fit_power_law(marg):
    """Fit power law to binned marginals."""
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals))
            bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p, bc, bm, intercept
    return None

# Compute curves
intact_ppl = compute_raw_ppl_curve(all_intact)
shuf_ppl = compute_raw_ppl_curve(all_shuffled)

# Marginals
intact_marg = -np.diff(intact_ppl)
shuf_marg = -np.diff(shuf_ppl)

# Corrected marginals
corrected_marg = intact_marg - shuf_marg

# Fit
result = fit_power_law(corrected_marg)
if result:
    slope, r, p, bc, bm, intercept = result
    print(f"FRENCH SPOKEN (corrected): alpha = {slope:.3f} (r = {r:.3f}, p = {p:.4f})")
else:
    print("WARNING: Power law fit failed")

# Uncorrected
norm_mean, norm_sem = compute_mean_curve(all_intact)
uncorr_marg = np.diff(norm_mean)
result_uncorr = fit_power_law(uncorr_marg)
if result_uncorr:
    print(f"FRENCH SPOKEN (uncorrected): alpha = {result_uncorr[0]:.3f} (r = {result_uncorr[1]:.3f})")

print(f"\nTotal perplexity drop: {intact_ppl[0]:.1f} -> {intact_ppl[-1]:.1f} "
      f"({(1 - intact_ppl[-1]/intact_ppl[0])*100:.1f}% reduction)")

In [ ]:
# === Figure 1: French standalone results (2x3 panel) ===
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
COLOR = '#8B008B'  # dark magenta for French

ax = axes[0, 0]
ax.plot(common_x, intact_ppl, '-', color=COLOR, linewidth=2, label='French intact')
ax.plot(common_x, shuf_ppl, ':', color=COLOR, linewidth=2, label='French shuffled')
ax.set_xlabel('Context Length (tokens)'); ax.set_ylabel('Perplexity')
ax.set_title('A. Raw Perplexity: Intact vs Shuffled', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

ax = axes[0, 1]
ax.plot(common_x[1:], uniform_filter1d(intact_marg, 5), '-', color=COLOR, linewidth=2, label='French intact')
ax.plot(common_x[1:], uniform_filter1d(shuf_marg, 5), ':', color=COLOR, linewidth=2, label='French shuffled')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)'); ax.set_ylabel('Marginal PPL Drop per Token')
ax.set_title('B. Marginal Gain: Intact vs Shuffled', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

ax = axes[0, 2]
ax.plot(common_x[1:], uniform_filter1d(corrected_marg, 5), '-', color=COLOR, linewidth=2, label='French (corrected)')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)'); ax.set_ylabel('Corrected Marginal (intact - shuffled)')
ax.set_title('C. Pure Coherence Signal', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.2)

ax = axes[1, 0]
result_fit = fit_power_law(corrected_marg)
if result_fit:
    slope, r, p, bc, bm, intercept = result_fit
    ax.plot(bc, bm, 'o-', color=COLOR, linewidth=2, markersize=6, label='French')
    fit_x = np.linspace(min(bc), max(bc), 100)
    fit_y = np.exp(intercept) * fit_x ** slope
    ax.plot(fit_x, fit_y, '--', color=COLOR, alpha=0.5,
            label=f'French: d^{slope:.2f} (r={r:.2f}, p={p:.4f})')
ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens, log)'); ax.set_ylabel('Corrected Marginal Benefit')
ax.set_title('D. Power Law Fit (Corrected)', fontweight='bold')
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)

ax = axes[1, 1]
for frac in TARGET_FRACTIONS:
    frac_curves = [c for c in all_intact if c['target_frac'] == frac]
    frac_shuf = [c for c in all_shuffled if c['target_frac'] == frac]
    if len(frac_curves) < 3: continue
    frac_intact_ppl = compute_raw_ppl_curve(frac_curves)
    frac_shuf_ppl = compute_raw_ppl_curve(frac_shuf)
    frac_corr = -np.diff(frac_intact_ppl) - (-np.diff(frac_shuf_ppl))
    frac_fit = fit_power_law(frac_corr)
    if frac_fit:
        s, r_val, p_val, fc_bc, fc_bm, fc_int = frac_fit
        ax.plot(fc_bc, fc_bm, 'o-', markersize=5, label=f'{frac:.0%}: d^{s:.2f} (r={r_val:.2f})')
ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens, log)'); ax.set_ylabel('Corrected Marginal')
ax.set_title('E. Stability: Exponent by Target Position', fontweight='bold')
ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

ax = axes[1, 2]
cum = np.cumsum(corrected_marg)
cum_norm = cum / cum[-1] if cum[-1] > 0 else cum
ax.plot(common_x[1:], cum_norm, '-', color=COLOR, linewidth=2, label='French')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)'); ax.set_ylabel('Fraction of Total Corrected Benefit')
ax.set_title('F. Cumulative Coherence Benefit', fontweight='bold')
half_idx = np.argmin(np.abs(cum_norm - 0.5)) + 1
ax.axvline(half_idx, color=COLOR, linestyle=':', alpha=0.5)
ax.text(half_idx + 2, 0.45, f'50% at {half_idx} tokens', fontsize=9, color=COLOR)
ax.legend(); ax.grid(True, alpha=0.2)

plt.suptitle('French Oral Narrative: Token-Level Coherence Decay',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_french_finegrain.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2: Cross-Language / Cross-Modality Comparison

Load RAID and Buckeye results and overlay all power law fits:
- **RAID written human** (English): alpha = -0.75
- **RAID written AI**: alpha = -1.97  
- **Buckeye spoken** (English): alpha = -0.73
- **French oral narrative**: alpha = ???
- **Anderson & Schooler (1991)**: -0.77

In [ ]:
# === Load RAID + Buckeye results for comparison ===

# RAID
raid_intact_path = DRIVE_RAID_RESULTS / "finegrain_results_v4.json"
raid_shuffled_path = DRIVE_RAID_RESULTS / "finegrain_shuffled_v4.json"
has_raid = raid_intact_path.exists() and raid_shuffled_path.exists()

if has_raid:
    with open(raid_intact_path) as f:
        raid_all = json.load(f)
    with open(raid_shuffled_path) as f:
        raid_all_shuf = json.load(f)
    raid_human = [c for c in raid_all if c['population'] == 'human']
    raid_ai = [c for c in raid_all if c['population'] == 'ai']
    raid_human_shuf = [c for c in raid_all_shuf if c['population'] == 'human']
    raid_ai_shuf = [c for c in raid_all_shuf if c['population'] == 'ai']
    rh_corr = -np.diff(compute_raw_ppl_curve(raid_human)) - (-np.diff(compute_raw_ppl_curve(raid_human_shuf)))
    ra_corr = -np.diff(compute_raw_ppl_curve(raid_ai)) - (-np.diff(compute_raw_ppl_curve(raid_ai_shuf)))
    print(f"RAID: {len(raid_human)} human, {len(raid_ai)} AI curves")
else:
    print(f"RAID results not found at {raid_intact_path}")

# Buckeye
bk_intact_path = DRIVE_BUCKEYE_RESULTS / "buckeye_intact_v1.json"
bk_shuffled_path = DRIVE_BUCKEYE_RESULTS / "buckeye_shuffled_v1.json"
has_buckeye = bk_intact_path.exists() and bk_shuffled_path.exists()

if has_buckeye:
    with open(bk_intact_path) as f:
        bk_all = json.load(f)
    with open(bk_shuffled_path) as f:
        bk_all_shuf = json.load(f)
    bk_corr = -np.diff(compute_raw_ppl_curve(bk_all)) - (-np.diff(compute_raw_ppl_curve(bk_all_shuf)))
    print(f"Buckeye: {len(bk_all)} curves")
else:
    print(f"Buckeye results not found at {bk_intact_path}")

In [ ]:
# === Figure 2: Cross-language / cross-modality comparison ===
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Collect all fits for the bar chart
all_fits = []  # (label, exponent, r, color, marker, marginals)

# French (this notebook)
fr_fit = fit_power_law(corrected_marg)
if fr_fit:
    all_fits.append(('French spoken\n(Oral Narrative)', fr_fit[0], fr_fit[1], '#8B008B', 'D', corrected_marg))

# Buckeye
if has_buckeye:
    bk_fit = fit_power_law(bk_corr)
    if bk_fit:
        all_fits.append(('English spoken\n(Buckeye)', bk_fit[0], bk_fit[1], 'green', 's', bk_corr))

# RAID human
if has_raid:
    rh_fit = fit_power_law(rh_corr)
    if rh_fit:
        all_fits.append(('English written\n(RAID human)', rh_fit[0], rh_fit[1], 'blue', 'o', rh_corr))
    ra_fit = fit_power_law(ra_corr)
    if ra_fit:
        all_fits.append(('English written\n(RAID AI)', ra_fit[0], ra_fit[1], 'red', '^', ra_corr))

# --- Panel A: Power law fits overlaid ---
ax = axes[0]
for label, slope, r, color, marker, marg in all_fits:
    fit = fit_power_law(marg)
    if fit:
        s, r_val, p, bc, bm, inter = fit
        ax.plot(bc, bm, f'{marker}-', color=color, linewidth=2, markersize=7,
                label=f'{label.replace(chr(10), " ")}: {s:.2f}')
        fit_x = np.linspace(1.5, 87, 100)
        ax.plot(fit_x, np.exp(inter) * fit_x**s, '--', color=color, alpha=0.3)
ax.set_xscale('log')
ax.set_xlabel('Context Distance (tokens)', fontsize=12)
ax.set_ylabel('Corrected Marginal Benefit', fontsize=12)
ax.set_title('A. Power Law Comparison', fontweight='bold', fontsize=13)
ax.legend(fontsize=8, title='Exponent (corrected)')
ax.grid(True, alpha=0.2)

# --- Panel B: Exponent bar chart ---
ax = axes[1]
labels = [f[0] for f in all_fits] + ['Anderson &\nSchooler (1991)']
exponents = [f[1] for f in all_fits] + [-0.77]
colors = [f[3] for f in all_fits] + ['gray']
r_vals = [f[2] for f in all_fits] + [None]

bars = ax.bar(range(len(labels)), exponents, color=colors, alpha=0.7, edgecolor='black')
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Power Law Exponent', fontsize=12)
ax.set_title('B. Decay Exponents: Cross-Language', fontweight='bold', fontsize=13)
ax.axhline(-0.77, color='gray', linestyle=':', alpha=0.5)
ax.grid(True, alpha=0.2, axis='y')
for i, (exp, r_val) in enumerate(zip(exponents, r_vals)):
    lbl = f'{exp:.2f}'
    if r_val is not None: lbl += f'\n(r={r_val:.2f})'
    ax.text(i, exp - 0.08, lbl, ha='center', fontsize=9, fontweight='bold')

# --- Panel C: Corrected marginals smoothed overlay ---
ax = axes[2]
for label, slope, r, color, marker, marg in all_fits:
    style = '--' if 'AI' in label else '-'
    ax.plot(common_x[1:], uniform_filter1d(marg, 5), style, color=color, linewidth=2,
            label=label.replace(chr(10), ' '))
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('Corrected Marginal', fontsize=12)
ax.set_title('C. Coherence Signal Across Languages', fontweight='bold', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

plt.suptitle('Cross-Language Coherence Decay: French vs English (Written & Spoken)',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig2_cross_language.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("CROSS-LANGUAGE COMPARISON")
print("="*60)
for label, exp in zip(labels, exponents):
    print(f"  {label.replace(chr(10), ' '):<35}: alpha = {exp:.3f}")

## Per-Storyteller Analysis

Check exponent stability across the 17 storytellers. Since we have multiple stories per teller, this gives per-individual estimates.

In [ ]:
# === Per-storyteller exponent analysis ===
COLOR = '#8B008B'

# Group by storyteller (author_id in the doc)
storyteller_map = {}
for doc in corpus:
    storyteller_map[doc['doc_id']] = doc['author_id']

# Aggregate curves by storyteller
from collections import defaultdict
teller_intact = defaultdict(list)
teller_shuffled = defaultdict(list)
for c in all_intact:
    tid = storyteller_map.get(c['doc_id'], c['doc_id'])
    teller_intact[tid].append(c)
for c in all_shuffled:
    tid = storyteller_map.get(c['doc_id'], c['doc_id'])
    teller_shuffled[tid].append(c)

teller_exponents = []
for tid in sorted(teller_intact.keys()):
    ti = teller_intact[tid]
    ts = teller_shuffled[tid]
    if len(ti) < 2 or len(ts) < 2:
        continue
    ti_ppl = compute_raw_ppl_curve(ti)
    ts_ppl = compute_raw_ppl_curve(ts)
    t_corr = -np.diff(ti_ppl) - (-np.diff(ts_ppl))
    t_fit = fit_power_law(t_corr)
    if t_fit:
        teller_exponents.append({
            'storyteller': tid,
            'exponent': t_fit[0],
            'r': t_fit[1],
            'p': t_fit[2],
            'n_curves': len(ti),
        })

if teller_exponents:
    df_t = pd.DataFrame(teller_exponents)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.hist(df_t['exponent'], bins=12, color=COLOR, alpha=0.7, edgecolor='black')
    ax.axvline(df_t['exponent'].mean(), color='darkmagenta', linestyle='-', linewidth=2,
               label=f'Mean: {df_t["exponent"].mean():.2f}')
    ax.axvline(df_t['exponent'].median(), color='darkmagenta', linestyle='--', linewidth=2,
               label=f'Median: {df_t["exponent"].median():.2f}')
    ax.axvline(-0.75, color='blue', linestyle=':', linewidth=2, label='RAID written: -0.75')
    ax.axvline(-0.73, color='green', linestyle=':', linewidth=2, label='Buckeye spoken: -0.73')
    ax.set_xlabel('Decay Exponent', fontsize=12); ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Per-Storyteller Exponents', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

    ax = axes[1]
    df_sorted = df_t.sort_values('exponent').reset_index(drop=True)
    ax.bar(range(len(df_sorted)), df_sorted['exponent'], color=COLOR, alpha=0.6)
    ax.axhline(df_t['exponent'].mean(), color='darkmagenta', linestyle='-', linewidth=2)
    ax.axhline(-0.75, color='blue', linestyle=':', linewidth=2, label='RAID written human')
    ax.axhline(-0.73, color='green', linestyle=':', linewidth=2, label='Buckeye spoken')
    ax.axhline(-1.97, color='red', linestyle=':', linewidth=2, label='RAID written AI')
    ax.set_xlabel('Storyteller (sorted)', fontsize=12); ax.set_ylabel('Decay Exponent', fontsize=12)
    ax.set_title('Per-Storyteller Exponents (sorted)', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(BASE_DIR / 'fig3_per_storyteller.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nPer-storyteller exponents (n={len(df_t)}):")
    print(f"  Mean:   {df_t['exponent'].mean():.3f} +/- {df_t['exponent'].std():.3f}")
    print(f"  Median: {df_t['exponent'].median():.3f}")
    print(f"  Range:  [{df_t['exponent'].min():.3f}, {df_t['exponent'].max():.3f}]")
    print(f"  Significant fits (p<0.05): {(df_t['p'] < 0.05).sum()} / {len(df_t)}")
else:
    print("Not enough per-storyteller data for individual fits")

## Summary

Cross-linguistic coherence decay results:

| Corpus | Language | Modality | Exponent |
|--------|----------|----------|----------|
| RAID (human) | English | Written | -0.75 |
| Buckeye | English | Spoken | -0.73 |
| **French Oral Narrative** | **French** | **Spoken** | **see above** |
| RAID (AI) | English | Written | -1.97 |
| Anderson & Schooler (1991) | English | Retrieval demand | -0.77 |

If the French exponent falls near -0.75, then:
1. The decay rate is **language-invariant** (not just English)
2. It's **modality-invariant** (written = spoken)
3. It matches memory retrieval statistics from independent research
4. It's a **universal property of human language production** constrained by working memory
5. AI text lacks this constraint entirely (exponent ~2x steeper)